# Phase 4 — Setup 1: Natural Heterogeneity

FedAvg · FedProx · Layer-by-Layer QPSO-FL

In [1]:
import os, sys, copy, time, random, json, shutil
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from PIL import Image
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
from scipy import stats
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42); device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu"); print("CPU only")
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

NUM_CLASSES = 4
CLASS_NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]
VALID_CLASSES = {"glioma": 0, "meningioma": 1, "notumor": 2, "no_tumor": 2, "pituitary": 3}

# ╔══════════════════════════════════════════════════════╗
# ║  Phase 4 Hyperparameters                            ║
# ╚══════════════════════════════════════════════════════╝
IMG_SIZE     = 112
LOCAL_EPOCHS = 1
LR           = 0.01
BATCH_SIZE   = 64
NUM_ROUNDS   = 100
FEDPROX_MU   = 0.01

# QPSO — faithful port from MNIST reference notebook
QPSO_BETA        = 0.6   # static contraction-expansion coefficient (same as MNIST)
QPSO_ITERS       = 5     # optimization iterations per layer per round (same default as MNIST)
QPSO_VAL_BATCHES = 4     # mini-batches used for each fitness evaluation (speed/quality tradeoff)

# Early Stopping (applies to all three methods independently)
PATIENCE   = 15     # stop if best accuracy does not improve for this many rounds
MIN_ROUNDS = 30     # always run at least this many rounds
TARGET_ACC = 95.0   # once this accuracy is reached, begin counting patience

print(f"Phase 4 | SimpleCNN | img={IMG_SIZE} | E={LOCAL_EPOCHS} | lr={LR} | bs={BATCH_SIZE} | max_rounds={NUM_ROUNDS}")
print(f"QPSO: beta={QPSO_BETA} (static) | iters/layer={QPSO_ITERS} | val_batches={QPSO_VAL_BATCHES}")
print(f"FedProx mu={FEDPROX_MU}")
print(f"Early Stopping: patience={PATIENCE} | min_rounds={MIN_ROUNDS} | target={TARGET_ACC}%")

for d in ['data/processed/client1','data/processed/client2','data/processed/client3',
          'data/test_set','models',
          'results_phase4/fedavg','results_phase4/fedprox','results_phase4/qpso',
          'results_phase4/plots','logs']:
    os.makedirs(f'/kaggle/working/{d}', exist_ok=True)
print("Directories created")


GPU: Tesla P100-PCIE-16GB
Phase 4 | SimpleCNN | img=112 | E=1 | lr=0.01 | bs=64 | max_rounds=100
QPSO: beta=0.6 (static) | iters/layer=5 | val_batches=4
FedProx mu=0.01
Early Stopping: patience=15 | min_rounds=30 | target=95.0%
Directories created


In [2]:
class Preprocessor:
    def __init__(self, size=None):
        self.size = size or (IMG_SIZE, IMG_SIZE)

    def load_image(self, path):
        try:
            img = Image.open(path)
            if img.mode != "RGB": img = img.convert("RGB")
            return np.array(img.resize(self.size, Image.LANCZOS), dtype=np.float32) / 255.0
        except: return None

    def process_dataset(self, dataset_path, client_name,
                        save_dir="/kaggle/working/data/processed",
                        train_r=0.70, val_r=0.15):
        print(f"\n{'='*60}\nProcessing {client_name} <- {dataset_path}\n{'='*60}")
        images, labels = [], []
        for cls_name in sorted(os.listdir(dataset_path)):
            cls_path = os.path.join(dataset_path, cls_name)
            if not os.path.isdir(cls_path): continue
            key = cls_name.lower().replace(" ", "")
            if key not in VALID_CLASSES: print(f"  Skipping: {cls_name}"); continue
            label = VALID_CLASSES[key]
            files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
            print(f"  {cls_name}: {len(files)} images (label={label})")
            for fname in tqdm(files, desc=f"    {cls_name}", leave=False):
                arr = self.load_image(os.path.join(cls_path, fname))
                if arr is not None: images.append(arr); labels.append(label)
        X, y = np.array(images), np.array(labels)
        print(f"  Total: {len(X)} | Classes: {np.bincount(y, minlength=NUM_CLASSES)}")
        test_r = 1.0 - train_r - val_r
        X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=test_r, stratify=y, random_state=42)
        val_adj = val_r / (train_r + val_r)
        X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=val_adj, stratify=y_tmp, random_state=42)
        print(f"  Train:{len(X_train)} Val:{len(X_val)} Test:{len(X_test)}")
        out = os.path.join(save_dir, client_name); os.makedirs(out, exist_ok=True)
        for name, arr in [("X_train",X_train),("y_train",y_train),("X_val",X_val),
                          ("y_val",y_val),("X_test",X_test),("y_test",y_test)]:
            np.save(os.path.join(out, f"{name}.npy"), arr)
        print(f"  Saved -> {out}")

    @staticmethod
    def create_global_test(pdir="/kaggle/working/data/processed", sdir="/kaggle/working/data/test_set"):
        Xs, ys = [], []
        for i in range(1, 4):
            d = os.path.join(pdir, f"client{i}")
            Xs.append(np.load(os.path.join(d, "X_test.npy")))
            ys.append(np.load(os.path.join(d, "y_test.npy")))
        X, y = np.concatenate(Xs), np.concatenate(ys)
        os.makedirs(sdir, exist_ok=True)
        np.save(os.path.join(sdir, "X_test.npy"), X); np.save(os.path.join(sdir, "y_test.npy"), y)
        print(f"Global test: {len(X)} images | Classes: {np.bincount(y, minlength=NUM_CLASSES)}")

print("Preprocessor ready")


Preprocessor ready


In [3]:
class BrainTumorDataset(Dataset):
    _DEFAULT = transforms.Compose([transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    def __init__(self, X, y, transform=None):
        self.X, self.y = X, y; self.transform = transform or self._DEFAULT
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        img = Image.fromarray((self.X[idx]*255).astype(np.uint8))
        return self.transform(img), torch.tensor(self.y[idx], dtype=torch.long)

TRAIN_TF = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5), transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
TEST_TF = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

def create_data_loaders(pdir="/kaggle/working/data/processed",
                        tdir="/kaggle/working/data/test_set", bs=None, nw=2):
    bs = bs or BATCH_SIZE; loaders = {}
    for i in range(1, 4):
        d = f"{pdir}/client{i}"
        ds_tr = BrainTumorDataset(np.load(f"{d}/X_train.npy"), np.load(f"{d}/y_train.npy"), TRAIN_TF)
        ds_va = BrainTumorDataset(np.load(f"{d}/X_val.npy"),   np.load(f"{d}/y_val.npy"),   TEST_TF)
        ds_te = BrainTumorDataset(np.load(f"{d}/X_test.npy"),  np.load(f"{d}/y_test.npy"),  TEST_TF)
        loaders[f"client{i}"] = {
            "train": DataLoader(ds_tr, bs, shuffle=True,  num_workers=nw, pin_memory=True),
            "val":   DataLoader(ds_va, bs, shuffle=False, num_workers=nw, pin_memory=True),
            "test":  DataLoader(ds_te, bs, shuffle=False, num_workers=nw, pin_memory=True),
            "train_size": len(ds_tr)}
        print(f"client{i}: Train={len(ds_tr)} Val={len(ds_va)} Test={len(ds_te)}")
    gds = BrainTumorDataset(np.load(f"{tdir}/X_test.npy"), np.load(f"{tdir}/y_test.npy"), TEST_TF)
    loaders["global_test"] = DataLoader(gds, bs, shuffle=False, num_workers=nw, pin_memory=True)
    print(f"Global test: {len(gds)}"); return loaders

# ═══════════════════════════════════════════════════════════════
# SimpleCNN — identical to Phase 3 (~120K params)
# ═══════════════════════════════════════════════════════════════
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(64*4*4, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0), -1))

def create_model(n=4, dev=None):
    dev = dev or device; m = SimpleCNN(n).to(dev)
    total = sum(p.numel() for p in m.parameters())
    print(f"SimpleCNN | params={total:,} | classes={n}"); return m

print("Dataset / model definitions ready")


Dataset / model definitions ready


In [4]:
# ═══════════════════════════════════════════════════════════════
# Federated Learning Clients & FedAvg Server
# ═══════════════════════════════════════════════════════════════
class FederatedClient:
    def __init__(self, cid, train_loader, val_loader, device="cuda"):
        self.client_id = cid; self.train_loader = train_loader
        self.val_loader = val_loader; self.device = device
        self.model = None; self.optimizer = None
        self.criterion = nn.CrossEntropyLoss()
        self.dataset_size = len(train_loader.dataset)

    def set_model(self, gm):      self.model = copy.deepcopy(gm).to(self.device)
    def set_optimizer(self, lr=None): self.optimizer = optim.Adam(self.model.parameters(), lr=lr or LR)
    def get_dataset_size(self):   return self.dataset_size

    def train_local(self, epochs=None, verbose=False, mu=None, global_params=None):
        epochs = epochs or LOCAL_EPOCHS; self.model.train(); el, ea = [], []
        for ep in range(epochs):
            ls, c, t = 0.0, 0, 0
            for imgs, labs in self.train_loader:
                imgs, labs = imgs.to(self.device), labs.to(self.device)
                self.optimizer.zero_grad(); out = self.model(imgs)
                loss = self.criterion(out, labs)
                if mu is not None and global_params is not None:
                    prox = sum(torch.norm(pl - pg.detach())**2
                               for pl, pg in zip(self.model.parameters(), global_params))
                    loss = loss + (mu / 2.0) * prox
                loss.backward(); self.optimizer.step()
                ls += loss.item(); _, pred = out.max(1)
                t += labs.size(0); c += pred.eq(labs).sum().item()
            el.append(ls/len(self.train_loader)); ea.append(100.*c/t)
        return self.model.state_dict(), el, ea

    def validate(self):
        self.model.eval(); ls, c, t = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labs in self.val_loader:
                imgs, labs = imgs.to(self.device), labs.to(self.device)
                out = self.model(imgs); ls += self.criterion(out, labs).item()
                _, pred = out.max(1); t += labs.size(0); c += pred.eq(labs).sum().item()
        return ls/len(self.val_loader), 100.*c/t

class FedProxClient(FederatedClient): pass   # train_local already handles mu/global_params

class FedAvgServer:
    def __init__(self, gm, clients, device="cuda"):
        self.global_model = gm; self.clients = clients; self.device = device
        self.total_samples = sum(c.get_dataset_size() for c in clients)
        print(f"FedAvg Server | clients={len(clients)} total_samples={self.total_samples}")

    def aggregate_weights(self, cw):
        first = cw[0][0]; total = sum(n for _, n in cw); agg = {}
        for k in first:
            if first[k].is_floating_point():
                acc = torch.zeros_like(first[k], dtype=torch.float32)
                for sd, nk in cw: acc += sd[k].float() * (nk / total)
                agg[k] = acc.to(first[k].dtype)
            else: agg[k] = first[k].clone()
        return agg

    def evaluate_global_model(self, tl):
        self.global_model.eval(); crit = nn.CrossEntropyLoss(); ls, c, t = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labs in tl:
                imgs, labs = imgs.to(self.device), labs.to(self.device)
                out = self.global_model(imgs); ls += crit(out, labs).item()
                _, pred = out.max(1); t += labs.size(0); c += pred.eq(labs).sum().item()
        return 100.*c/t, ls/len(tl)

    def get_global_model(self): return copy.deepcopy(self.global_model)

print("FL client / FedAvg server classes loaded")


FL client / FedAvg server classes loaded


In [5]:
# ═══════════════════════════════════════════════════════════════
# Phase 4 QPSOServer — Layer-by-Layer QPSO
#
# Faithfully ported from the MNIST IID reference notebook.
#
# Algorithm (per round):
#   1. Collect client state_dicts (particles).
#   2. Compute FedAvg baseline (used for BN buffers + initial context).
#   3. For each trainable parameter tensor (layer) independently:
#        a. Positions = client weights for that layer, flattened.
#        b. Evaluate initial fitness (val LOSS) for each particle.
#        c. Run QPSO_ITERS update steps:
#             mbest = mean of personal bests
#             For each particle i:
#               phi  = rand(dim)
#               u    = rand(dim) + epsilon          (no hard clamp, raw random)
#               sign = random ±1 per dimension
#               p    = phi*pbest_i + (1-phi)*gbest  (attractor)
#               step = beta * |mbest - x_i| * log(1/u) * sign
#               new_pos = p + step                  (matches MNIST formula exactly)
#               Evaluate fitness; update pbest/gbest if loss improved.
#        d. Set this layer = gbest.
#   4. Assembled model = all layers from step 3d.
# ═══════════════════════════════════════════════════════════════
class QPSOServer:
    def __init__(self, gm, clients, device="cuda",
                 beta=None, q_iters=None, val_loader=None, val_batches=None):
        self.global_model = gm
        self.clients      = clients
        self.device       = device
        self.beta         = beta       or QPSO_BETA
        self.q_iters      = q_iters    or QPSO_ITERS
        self.val_loader   = val_loader
        self.val_batches  = val_batches or QPSO_VAL_BATCHES
        self._param_names = {n for n, _ in gm.named_parameters()}
        print(f"QPSOServer | clients={len(clients)} | beta={self.beta} | "
              f"iters/layer={self.q_iters} | trainable_tensors={len(self._param_names)}")
        if val_loader is None:
            print("  WARNING: val_loader not set — call set_val_loader() before training")

    def set_val_loader(self, loader): self.val_loader = loader

    # ── Fitness: validation loss with one layer substituted ──────────────────
    # Minimise loss — lower is better (matches MNIST reference).
    def _eval_loss(self, base_sd, layer_key, candidate_flat, layer_shape):
        sd = copy.deepcopy(base_sd)
        sd[layer_key] = candidate_flat.reshape(layer_shape).to(base_sd[layer_key].dtype)
        self.global_model.load_state_dict(sd)
        self.global_model.eval()
        crit = nn.CrossEntropyLoss(); total = 0.0; nb = 0
        with torch.no_grad():
            for imgs, labs in self.val_loader:
                imgs, labs = imgs.to(self.device), labs.to(self.device)
                total += crit(self.global_model(imgs), labs).item()
                nb += 1
                if nb >= self.val_batches: break
        return total / max(nb, 1)

    # ── FedAvg helper — baseline for non-trainable buffers ───────────────────
    def _fedavg(self, client_weights):
        first = client_weights[0]; n = len(client_weights); agg = {}
        for k in first:
            if first[k].is_floating_point():
                acc = torch.zeros_like(first[k], dtype=torch.float32)
                for w in client_weights: acc += w[k].float()
                agg[k] = (acc / n).to(first[k].dtype)
            else: agg[k] = first[k].clone()
        return agg

    # ── Main layer-by-layer aggregation ──────────────────────────────────────
    def qpso_aggregate(self, client_weights, rnd=1):
        assert self.val_loader is not None, "val_loader must be set before qpso_aggregate()"
        n_particles = len(client_weights)

        # FedAvg baseline — valid state for the whole model at all times
        base_sd      = self._fedavg(client_weights)
        optimized_sd = copy.deepcopy(base_sd)   # will be updated layer by layer

        for k in base_sd:
            if k not in self._param_names or not base_sd[k].is_floating_point():
                continue   # skip BN running_mean / running_var / num_batches_tracked

            shape = base_sd[k].shape

            # Initialise particle positions from client weights for this layer
            positions = torch.stack(
                [w[k].float().cpu().flatten() for w in client_weights])  # (n_particles, dim)
            dim = positions.shape[1]

            pbest        = positions.clone()
            pbest_scores = torch.zeros(n_particles)

            # Evaluate initial fitness for each particle
            for i in range(n_particles):
                pbest_scores[i] = self._eval_loss(optimized_sd, k, pbest[i], shape)

            gbest_idx   = int(pbest_scores.argmin())
            gbest       = pbest[gbest_idx].clone()
            gbest_score = pbest_scores[gbest_idx].item()

            print(f"  [Layer {k}] dim={dim:,}  init_best_loss={gbest_score:.4f}")

            # QPSO iterations for this layer
            for it in range(self.q_iters):
                mbest = pbest.mean(dim=0)    # mean best position
                for i in range(n_particles):
                    phi  = torch.rand(dim)
                    # u: raw random in (0,1], no hard clamp — matches MNIST reference
                    u    = torch.rand(dim).clamp(min=1e-9)
                    sign = torch.where(torch.rand(dim) > 0.5,
                                       torch.ones(dim), -torch.ones(dim))

                    # Attractor: convex combination of pbest_i and gbest
                    p = phi * pbest[i] + (1.0 - phi) * gbest

                    # Quantum step (no perturbation clamp — matches MNIST reference)
                    step    = self.beta * torch.abs(mbest - positions[i]) * torch.log(1.0 / u) * sign
                    new_pos = p + step        # MNIST formula: new_pos = p + step

                    positions[i] = new_pos
                    score = self._eval_loss(optimized_sd, k, new_pos, shape)

                    if score < pbest_scores[i]:   # minimise loss
                        pbest_scores[i] = score
                        pbest[i]        = new_pos.clone()

                    if score < gbest_score:
                        gbest_score = score
                        gbest       = new_pos.clone()

                print(f"    iter {it+1}/{self.q_iters}  best_loss={gbest_score:.4f}")

            # Commit gbest for this layer into the running optimized state
            optimized_sd[k] = gbest.reshape(shape).to(base_sd[k].dtype)

        self.global_model.load_state_dict(optimized_sd)
        print("  QPSO aggregation complete")
        return optimized_sd

    def evaluate_global_model(self, tl):
        self.global_model.eval(); crit = nn.CrossEntropyLoss(); ls, c, t = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labs in tl:
                imgs, labs = imgs.to(self.device), labs.to(self.device)
                out = self.global_model(imgs); ls += crit(out, labs).item()
                _, pred = out.max(1); t += labs.size(0); c += pred.eq(labs).sum().item()
        return 100.*c/t, ls/len(tl)

    def get_global_model(self): return copy.deepcopy(self.global_model)

print("QPSOServer (layer-by-layer, Phase 4) loaded")


QPSOServer (layer-by-layer, Phase 4) loaded


In [6]:
# ═══════════════════════════════════════════════════════════════
# Training loops with Early Stopping
#
# Early stopping applies to all three methods identically:
#   - Always run at least MIN_ROUNDS rounds.
#   - Once global test accuracy >= TARGET_ACC, start patience countdown.
#   - Stop when best accuracy hasn't improved for PATIENCE consecutive rounds.
# ═══════════════════════════════════════════════════════════════

def _early_stop_check(rnd, best_acc, acc, no_improve_streak):
    # Returns (updated_best, updated_streak, should_stop).
    if acc > best_acc:
        return acc, 0, False
    if rnd < MIN_ROUNDS:
        return best_acc, no_improve_streak + 1, False
    if best_acc >= TARGET_ACC:
        streak = no_improve_streak + 1
        if streak >= PATIENCE:
            return best_acc, streak, True
        return best_acc, streak, False
    return best_acc, no_improve_streak + 1, False


def train_fedavg(server, clients, gtl, nr=None, le=None, lr=None,
                 se=10, sd="/kaggle/working", verbose=False):
    nr = nr or NUM_ROUNDS; le = le or LOCAL_EPOCHS; lr = lr or LR
    print("="*80 + f"\n  FEDAVG  rounds={nr} | epochs={le} | lr={lr}\n" + "="*80)
    h = {k:[] for k in ["round","global_test_acc","global_test_loss",
                         "client1_val_acc","client2_val_acc","client3_val_acc","round_time"]}
    best = 0.0; streak = 0
    for rnd in range(1, nr+1):
        t0 = time.time(); cw = []
        for cl in clients:
            cl.set_model(server.get_global_model()); cl.set_optimizer(lr)
            w,_,_ = cl.train_local(le, verbose=verbose)
            _, va = cl.validate()
            h[f"{cl.client_id}_val_acc"].append(va); cw.append((w, cl.get_dataset_size()))
        agg = server.aggregate_weights(cw); server.global_model.load_state_dict(agg)
        ga, gl = server.evaluate_global_model(gtl); dt = time.time()-t0
        h["round"].append(rnd); h["global_test_acc"].append(ga)
        h["global_test_loss"].append(gl); h["round_time"].append(dt)
        tag = ""
        if ga > best: torch.save(server.global_model.state_dict(), f"{sd}/models/fedavg_best.pth"); tag = " *** best"
        if rnd % se == 0: torch.save(server.global_model.state_dict(), f"{sd}/models/fedavg_round_{rnd}.pth")
        pd.DataFrame(h).to_csv(f"{sd}/results_phase4/fedavg/metrics.csv", index=False)
        print(f"  R{rnd:3d} | acc={ga:.2f}% loss={gl:.4f} ({dt:.1f}s){tag}")
        best, streak, stop = _early_stop_check(rnd, best, ga, streak)
        if stop: print(f"  Early stop at round {rnd} (patience={PATIENCE}, best={best:.2f}%)"); break
    print(f"  FedAvg done — best {best:.2f}%"); return h


def train_fedprox(server, clients, gtl, nr=None, le=None, lr=None, mu=None,
                  se=10, sd="/kaggle/working", verbose=False):
    nr = nr or NUM_ROUNDS; le = le or LOCAL_EPOCHS; lr = lr or LR; mu = mu or FEDPROX_MU
    print("="*80 + f"\n  FEDPROX  rounds={nr} | epochs={le} | lr={lr} | mu={mu}\n" + "="*80)
    h = {k:[] for k in ["round","global_test_acc","global_test_loss",
                         "client1_val_acc","client2_val_acc","client3_val_acc","round_time"]}
    best = 0.0; streak = 0
    for rnd in range(1, nr+1):
        t0 = time.time(); cw = []; gp = list(server.global_model.parameters())
        for cl in clients:
            cl.set_model(server.get_global_model()); cl.set_optimizer(lr)
            w,_,_ = cl.train_local(le, verbose=verbose, mu=mu, global_params=gp)
            _, va = cl.validate()
            h[f"{cl.client_id}_val_acc"].append(va); cw.append((w, cl.get_dataset_size()))
        agg = server.aggregate_weights(cw); server.global_model.load_state_dict(agg)
        ga, gl = server.evaluate_global_model(gtl); dt = time.time()-t0
        h["round"].append(rnd); h["global_test_acc"].append(ga)
        h["global_test_loss"].append(gl); h["round_time"].append(dt)
        tag = ""
        if ga > best: torch.save(server.global_model.state_dict(), f"{sd}/models/fedprox_best.pth"); tag = " *** best"
        if rnd % se == 0: torch.save(server.global_model.state_dict(), f"{sd}/models/fedprox_round_{rnd}.pth")
        pd.DataFrame(h).to_csv(f"{sd}/results_phase4/fedprox/metrics.csv", index=False)
        print(f"  R{rnd:3d} | acc={ga:.2f}% loss={gl:.4f} ({dt:.1f}s){tag}")
        best, streak, stop = _early_stop_check(rnd, best, ga, streak)
        if stop: print(f"  Early stop at round {rnd} (patience={PATIENCE}, best={best:.2f}%)"); break
    print(f"  FedProx done — best {best:.2f}%"); return h


def train_qpso(server, clients, gtl, nr=None, le=None, lr=None,
               se=10, sd="/kaggle/working", verbose=False):
    nr = nr or NUM_ROUNDS; le = le or LOCAL_EPOCHS; lr = lr or LR
    print("="*80 + f"\n  QPSO-FL (layer-by-layer)  rounds={nr} | beta={server.beta} | iters/layer={server.q_iters}\n" + "="*80)
    h = {k:[] for k in ["round","global_test_acc","global_test_loss",
                         "client1_val_acc","client2_val_acc","client3_val_acc","round_time"]}
    best = 0.0; streak = 0
    for rnd in range(1, nr+1):
        t0 = time.time()
        print(f"\n{'='*60} ROUND {rnd}/{nr}")
        cws = []
        for cl in clients:
            cl.set_model(server.get_global_model()); cl.set_optimizer(lr)
            w,_,_ = cl.train_local(le, verbose=verbose)
            _, va = cl.validate(); print(f"  {cl.client_id} val={va:.2f}%")
            h[f"{cl.client_id}_val_acc"].append(va); cws.append(w)
        print(f"  QPSO aggregating (beta={server.beta}, iters/layer={server.q_iters})...")
        agg = server.qpso_aggregate(cws, rnd)
        server.global_model.load_state_dict(agg)
        ga, gl = server.evaluate_global_model(gtl); dt = time.time()-t0
        h["round"].append(rnd); h["global_test_acc"].append(ga)
        h["global_test_loss"].append(gl); h["round_time"].append(dt)
        tag = ""
        if ga > best: torch.save(server.global_model.state_dict(), f"{sd}/models/qpso_best.pth"); tag = " *** best"
        if rnd % se == 0: torch.save(server.global_model.state_dict(), f"{sd}/models/qpso_round_{rnd}.pth")
        pd.DataFrame(h).to_csv(f"{sd}/results_phase4/qpso/metrics.csv", index=False)
        print(f"  Global: acc={ga:.2f}% loss={gl:.4f} ({dt:.1f}s){tag}")
        best, streak, stop = _early_stop_check(rnd, best, ga, streak)
        if stop: print(f"  Early stop at round {rnd} (patience={PATIENCE}, best={best:.2f}%)"); break
    print(f"  QPSO-FL done — best {best:.2f}%"); return h

print("Training functions loaded")


Training functions loaded


In [7]:
def plot_accuracy_comparison(dfs, names, save_path):
    colors = ['#1f77b4','#ff7f0e','#2ca02c']; n = len(dfs)
    fig, axes = plt.subplots(2, max(n,2), figsize=(8*max(n,2), 12))
    fig.suptitle("Phase 4: FL Aggregation Comparison (SimpleCNN)", fontsize=16, fontweight="bold")
    ax = axes[0,0]
    for df, name, c in zip(dfs, names, colors):
        ax.plot(df["round"], df["global_test_acc"], label=name, lw=2, color=c)
    ax.set(xlabel="Round", ylabel="Acc (%)", title="Global Test Accuracy", ylim=[0,100]); ax.legend(); ax.grid(alpha=0.3)
    ax = axes[0,1]
    for df, name, c in zip(dfs, names, colors):
        ax.plot(df["round"], df["global_test_loss"], label=name, lw=2, color=c)
    ax.set(xlabel="Round", ylabel="Loss", title="Global Test Loss"); ax.legend(); ax.grid(alpha=0.3)
    for idx, (df, name) in enumerate(zip(dfs, names)):
        ax = axes[1, idx]
        for i in range(1,4): ax.plot(df["round"], df[f"client{i}_val_acc"], label=f"Client {i}", lw=2)
        ax.set(xlabel="Round", ylabel="Val Acc (%)", title=f"{name}: Per-Client Val"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches="tight"); plt.show()
    print(f"Saved -> {save_path}")

def generate_confusion_matrix(mp, tl, mname, dev=None, save_path=None):
    dev = dev or device; model = SimpleCNN(NUM_CLASSES).to(dev)
    model.load_state_dict(torch.load(mp, map_location=dev, weights_only=True))
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for imgs, labs in tl:
            _, pred = model(imgs.to(dev)).max(1)
            preds.extend(pred.cpu().numpy()); labels.extend(labs.numpy())
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"Confusion Matrix — {mname}", fontsize=14, fontweight="bold")
    plt.ylabel("True"); plt.xlabel("Predicted"); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"\n{mname}:"); print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))

def plot_roc_auc(mps, mnames, tl, dev=None, save_path=None):
    dev = dev or device; n = len(mps)
    fig, axes = plt.subplots(1, n, figsize=(7*n, 6))
    if n == 1: axes = [axes]
    cc = ['#e74c3c','#3498db','#2ecc71','#9b59b6']
    for ax, mp, mn in zip(axes, mps, mnames):
        model = SimpleCNN(NUM_CLASSES).to(dev)
        model.load_state_dict(torch.load(mp, map_location=dev, weights_only=True))
        model.eval(); probs, labs = [], []
        with torch.no_grad():
            for imgs, lb in tl:
                out = model(imgs.to(dev))
                probs.append(torch.softmax(out,1).cpu().numpy()); labs.extend(lb.numpy())
        yp = np.vstack(probs); yt = np.array(labs)
        if np.isnan(yp).any(): yp = np.nan_to_num(yp, nan=1.0/NUM_CLASSES)
        yb = label_binarize(yt, classes=list(range(NUM_CLASSES)))
        for i, (cls, c) in enumerate(zip(CLASS_NAMES, cc)):
            try:
                fpr, tpr, _ = roc_curve(yb[:,i], yp[:,i]); a = auc(fpr,tpr)
                ax.plot(fpr, tpr, color=c, lw=2, label=f"{cls} ({a:.3f})")
            except: ax.plot([0,1],[0,1], color=c, lw=2, label=f"{cls} (N/A)")
        try:
            fpr_m, tpr_m, _ = roc_curve(yb.ravel(), yp.ravel())
            ax.plot(fpr_m, tpr_m, 'k--', lw=2, label=f"Micro ({auc(fpr_m,tpr_m):.3f})")
        except: pass
        ax.plot([0,1],[0,1],'gray',ls=':',lw=1)
        ax.set(xlabel="FPR", ylabel="TPR", title=f"ROC — {mn}"); ax.legend(loc="lower right", fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

def plot_client_fairness(dfs, names, save_path):
    n = len(dfs); cc = ["#FF6B6B","#4ECDC4","#45B7D1"]
    fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
    if n == 1: axes = [axes]
    for ax, df, t in zip(axes, dfs, names):
        accs = [df[f"client{i}_val_acc"].iloc[-1] for i in range(1,4)]
        ax.bar(["C1","C2","C3"], accs, color=cc, edgecolor="black")
        ax.axhline(np.mean(accs), color="red", ls="--", lw=2, label="Mean")
        ax.set(ylabel="Final Val Acc (%)", ylim=[0,100])
        ax.set_title(f"{t} (std={np.std(accs):.2f}%)", fontsize=13, fontweight="bold")
        ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches="tight"); plt.show()

def convergence_metrics(df, target=80.0):
    r = df[df["global_test_acc"] >= target]
    return {"final_acc":        round(df["global_test_acc"].iloc[-1], 2),
            "best_acc":         round(df["global_test_acc"].max(), 2),
            "rounds_run":       int(df["round"].max()),
            "round_to_target":  int(r["round"].min()) if len(r) else None,
            "avg_round_time_s": round(df["round_time"].mean(), 2),
            "total_time_min":   round(df["round_time"].sum() / 60, 2)}

def statistical_analysis(dfa, dfb):
    a, b = dfa["global_test_acc"].values, dfb["global_test_acc"].values
    n = min(len(a), len(b)); a, b = a[:n], b[:n]
    t_stat, p_val = stats.ttest_rel(b, a)
    d = (b-a).mean() / ((b-a).std(ddof=1) + 1e-8)
    return {"t_stat": round(float(t_stat),4), "p_value": float(p_val),
            "cohens_d": round(float(d),4), "significant": bool(p_val < 0.05)}

def build_comparison_df(*pairs):
    cols = ["Metric"] + [n for _, n in pairs]; ml = []
    for df, n in pairs:
        m = convergence_metrics(df)
        m["client_std"] = round(float(np.std([df[f"client{i}_val_acc"].iloc[-1] for i in range(1,4)])), 2)
        ml.append(m)
    keys = [("Final Acc (%)","final_acc"),("Best Acc (%)","best_acc"),
            ("Rounds Run","rounds_run"),("Rounds to 80%","round_to_target"),
            ("Avg Round (s)","avg_round_time_s"),("Total Time (min)","total_time_min"),
            ("Client Std Dev","client_std")]
    rows = [[l]+[m[k] if m[k] is not None else "N/A" for m in ml] for l, k in keys]
    return pd.DataFrame(rows, columns=cols)

print("Evaluation helpers loaded")


Evaluation helpers loaded


In [8]:
# ── Setup 1: Natural Heterogeneity ─────────────────────────────────────────
CLIENT1_PATH = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
CLIENT2_PATH = '/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task/train'
CLIENT3_PATH = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'

for name, path in [("Client1 (Masoud Test)", CLIENT1_PATH),
                   ("Client2 (BRISC)",        CLIENT2_PATH),
                   ("Client3 (Masoud Train)", CLIENT3_PATH)]:
    print(f"\n{name}: {path}")
    if os.path.exists(path):
        for d in sorted(os.listdir(path)):
            full = os.path.join(path, d)
            if os.path.isdir(full): print(f"  {d}/ ({len(os.listdir(full))} files)")
    else: print("  PATH NOT FOUND!")

prep = Preprocessor()
prep.process_dataset(CLIENT1_PATH, 'client1')
prep.process_dataset(CLIENT2_PATH, 'client2')
prep.process_dataset(CLIENT3_PATH, 'client3')
Preprocessor.create_global_test()
data_loaders = create_data_loaders()

# Combined val loader for QPSO fitness evaluation (no test-set leakage)
_val_ds = ConcatDataset([data_loaders[f"client{i}"]["val"].dataset for i in range(1,4)])
qpso_val_loader = DataLoader(_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"QPSO val loader: {len(_val_ds)} samples")
print("Setup 1 data ready!")



Client1 (Masoud Test): /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing
  glioma/ (400 files)
  meningioma/ (400 files)
  notumor/ (400 files)
  pituitary/ (400 files)

Client2 (BRISC): /kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task/train
  glioma/ (1147 files)
  meningioma/ (1329 files)
  no_tumor/ (1067 files)
  pituitary/ (1457 files)

Client3 (Masoud Train): /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training
  glioma/ (1400 files)
  meningioma/ (1400 files)
  notumor/ (1400 files)
  pituitary/ (1400 files)

Processing client1 <- /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing
  glioma: 400 images (label=0)


    glioma:   0%|          | 0/400 [00:00<?, ?it/s]

  meningioma: 400 images (label=1)


    meningioma:   0%|          | 0/400 [00:00<?, ?it/s]

  notumor: 400 images (label=2)


...

  notumor: 1400 images (label=2)


    notumor:   0%|          | 0/1400 [00:00<?, ?it/s]

  pituitary: 1400 images (label=3)


    pituitary:   0%|          | 0/1400 [00:00<?, ?it/s]

  Total: 5600 | Classes: [1400 1400 1400 1400]
  Train:3919 Val:840 Test:841
  Saved -> /kaggle/working/data/processed/client3
Global test: 1833 images | Classes: [442 470 432 489]
client1: Train=1119 Val=240 Test=241
client2: Train=3499 Val=750 Test=751
client3: Train=3919 Val=840 Test=841
Global test: 1833
QPSO val loader: 1830 samples
Setup 1 data ready!


In [9]:
# FedAvg
fedavg_model   = create_model(NUM_CLASSES, device)
fedavg_clients = [FederatedClient(f'client{i}', data_loaders[f'client{i}']['train'],
                  data_loaders[f'client{i}']['val'], device=device) for i in range(1,4)]
fedavg_server  = FedAvgServer(fedavg_model, fedavg_clients, device=device)
fedavg_history = train_fedavg(fedavg_server, fedavg_clients, data_loaders['global_test'],
                               nr=NUM_ROUNDS, le=LOCAL_EPOCHS, lr=LR, se=10)
torch.cuda.empty_cache()

# FedProx
fedprox_model   = create_model(NUM_CLASSES, device)
fedprox_clients = [FedProxClient(f'client{i}', data_loaders[f'client{i}']['train'],
                   data_loaders[f'client{i}']['val'], device=device) for i in range(1,4)]
fedprox_server  = FedAvgServer(fedprox_model, fedprox_clients, device=device)
fedprox_history = train_fedprox(fedprox_server, fedprox_clients, data_loaders['global_test'],
                                 nr=NUM_ROUNDS, le=LOCAL_EPOCHS, lr=LR, mu=FEDPROX_MU, se=10)
torch.cuda.empty_cache()

# QPSO-FL (layer-by-layer, loss-minimizing)
qpso_model   = create_model(NUM_CLASSES, device)
qpso_clients = [FederatedClient(f'client{i}', data_loaders[f'client{i}']['train'],
                data_loaders[f'client{i}']['val'], device=device) for i in range(1,4)]
qpso_server  = QPSOServer(qpso_model, qpso_clients, device=device,
                           beta=QPSO_BETA, q_iters=QPSO_ITERS,
                           val_loader=qpso_val_loader, val_batches=QPSO_VAL_BATCHES)
qpso_history = train_qpso(qpso_server, qpso_clients, data_loaders['global_test'],
                           nr=NUM_ROUNDS, le=LOCAL_EPOCHS, lr=LR, se=10)
torch.cuda.empty_cache()

print("\nAll Phase 4 Setup 1 training complete!")


SimpleCNN | params=155,524 | classes=4
FedAvg Server | clients=3 total_samples=8537
  FEDAVG  rounds=100 | epochs=1 | lr=0.01
  R  1 | acc=27.55% loss=1.3494 (8.9s) *** best
  R  2 | acc=56.68% loss=0.9751 (8.0s) *** best
  R  3 | acc=62.96% loss=0.8814 (8.0s) *** best
  R  4 | acc=76.98% loss=0.6204 (8.4s) *** best
  R  5 | acc=70.59% loss=0.7778 (7.9s)
  R  6 | acc=72.83% loss=0.6239 (7.9s)
  R  7 | acc=79.16% loss=0.5730 (7.8s) *** best
  R  8 | acc=78.12% loss=0.5733 (7.8s)
  R  9 | acc=78.34% loss=0.5508 (7.9s)
  R 10 | acc=79.76% loss=0.5439 (7.9s) *** best
  R 11 | acc=83.25% loss=0.4729 (7.8s) *** best
  R 12 | acc=79.21% loss=0.5234 (7.9s)
  R 13 | acc=84.62% loss=0.4393 (7.9s) *** best
  R 14 | acc=84.40% loss=0.4369 (7.9s)
  R 15 | acc=85.71% loss=0.4072 (7.7s) *** best
  R 16 | acc=84.78% loss=0.4042 (7.7s)
  R 17 | acc=85.43% loss=0.3887 (7.7s)
  R 18 | acc=87.34% loss=0.3592 (7.8s) *** best
  R 19 | acc=85.65% loss=0.3841 (7.8s)
  R 20 | acc=88.00% loss=0.3269 (7.8s) *** 

In [10]:
df_fa = pd.read_csv('/kaggle/working/results_phase4/fedavg/metrics.csv')
df_fp = pd.read_csv('/kaggle/working/results_phase4/fedprox/metrics.csv')
df_qp = pd.read_csv('/kaggle/working/results_phase4/qpso/metrics.csv')

comp = build_comparison_df((df_fa,"FedAvg"),(df_fp,"FedProx"),(df_qp,"QPSO-FL"))
print(comp.to_string(index=False))
comp.to_csv('/kaggle/working/results_phase4/final_comparison.csv', index=False)

plot_accuracy_comparison([df_fa, df_fp, df_qp], ['FedAvg','FedProx','QPSO-FL'],
                         '/kaggle/working/results_phase4/plots/comparison.png')

for m, n in [('fedavg','FedAvg'),('fedprox','FedProx'),('qpso','QPSO-FL')]:
    generate_confusion_matrix(f'/kaggle/working/models/{m}_best.pth',
                              data_loaders['global_test'], n,
                              save_path=f'/kaggle/working/results_phase4/plots/cm_{m}.png')

plot_client_fairness([df_fa, df_fp, df_qp], ['FedAvg','FedProx','QPSO-FL'],
                     '/kaggle/working/results_phase4/plots/fairness.png')

plot_roc_auc([f'/kaggle/working/models/{m}_best.pth' for m in ('fedavg','fedprox','qpso')],
             ['FedAvg','FedProx','QPSO-FL'],
             data_loaders['global_test'],
             save_path='/kaggle/working/results_phase4/plots/roc_auc.png')

s1 = statistical_analysis(df_fa, df_qp)
s2 = statistical_analysis(df_fa, df_fp)
print("FedAvg vs QPSO:",    json.dumps(s1, indent=2))
print("FedAvg vs FedProx:", json.dumps(s2, indent=2))

summary = {
    "phase": "Phase 4",
    "fedavg":  convergence_metrics(df_fa),
    "fedprox": convergence_metrics(df_fp),
    "qpso":    convergence_metrics(df_qp),
    "stats_fedavg_vs_qpso":   s1,
    "stats_fedavg_vs_fedprox": s2,
    "config": {
        "model": "SimpleCNN (~120K params)", "img_size": IMG_SIZE,
        "local_epochs": LOCAL_EPOCHS, "lr": LR, "batch_size": BATCH_SIZE,
        "num_rounds": NUM_ROUNDS, "fedprox_mu": FEDPROX_MU,
        "qpso_beta": QPSO_BETA, "qpso_iters_per_layer": QPSO_ITERS,
        "qpso_val_batches": QPSO_VAL_BATCHES,
        "early_stopping": {"patience": PATIENCE, "min_rounds": MIN_ROUNDS, "target_acc": TARGET_ACC},
        "qpso_type": "layer-by-layer, loss-minimizing, ported from MNIST IID reference"
    }
}
with open('/kaggle/working/results_phase4/executive_summary.json','w') as f:
    json.dump(summary, f, indent=2)
print("Summary saved")

print("\nLaTeX Table:")
print(comp.to_latex(index=False, float_format="%.2f",
      caption="Phase 4: FedAvg vs FedProx vs QPSO-FL (SimpleCNN, layer-by-layer)",
      label="tab:phase4_comparison"))
print("\nNotebook complete!")


          Metric  FedAvg  FedProx  QPSO-FL
   Final Acc (%)   93.29    95.85    93.78
    Best Acc (%)   95.80    96.13    95.14
      Rounds Run   98.00   100.00   100.00
   Rounds to 80%   11.00    12.00     9.00
   Avg Round (s)    7.84     8.03    75.09
Total Time (min)   12.81    13.39   125.14
  Client Std Dev    5.28     2.27     1.56
Saved -> /kaggle/working/results_phase4/plots/comparison.png

FedAvg:
              precision    recall  f1-score   support

      Glioma     0.9782    0.9118    0.9438       442
  Meningioma     0.9150    0.9617    0.9378       470
    No Tumor     0.9697    0.9630    0.9663       432
   Pituitary     0.9739    0.9918    0.9828       489

    accuracy                         0.9580      1833
   macro avg     0.9592    0.9571    0.9577      1833
weighted avg     0.9588    0.9580    0.9580      1833


FedProx:
              precision    recall  f1-score   support

      Glioma     0.9667    0.9186    0.9420       442
  Meningioma     0.9085    0.972